# Advanced RSVP Tools Experiments

This notebook provides richer visualizations and experimental tooling:
- Hyperparameter sweeps for TARTAN
- 2‑D & 3‑D trajectory visualizations
- Lamphrodynamic energy heatmaps
- Comparison between TARTAN and non‑TARTAN trajectories
- Interactive widgets (if available)


In [ ]:
import torch
from torch import nn
import matplotlib.pyplot as plt
import numpy as np

from rsvp_tools import (
    TARTANLayer, TARTANConfig,
    LamFlowWrapper, LamFlowConfig
)

print('Advanced notebook loaded.')

## Synthetic Dynamics Generator

In [ ]:
def generate_spiral_trajectory(T=200, noise=0.05):
    t = torch.linspace(0, 4 * torch.pi, T)
    x = t * torch.cos(t) + noise * torch.randn(T)
    y = t * torch.sin(t) + noise * torch.randn(T)
    z = torch.stack([x, y], dim=-1)
    return z

traj2d = generate_spiral_trajectory()
traj2d.shape

## TARTAN Hyperparameter Sweep

In [ ]:
sigmas = [0.2, 0.5, 1.0, 2.0]
contractions = [0.05, 0.1, 0.2]

results = {}

for sigma in sigmas:
    for c in contractions:
        cfg = TARTANConfig(aura_sigma=sigma, contraction_strength=c, buffer_size=64)
        layer = TARTANLayer(cfg)
        out = layer(traj2d.unsqueeze(1))[:,0,:].detach()
        results[(sigma, c)] = out

list(results.keys())[:5]

## Visualization: TARTAN Hyperparameter Effects

In [ ]:
plt.figure(figsize=(12, 10))
i = 1
for sigma in sigmas:
    for c in contractions:
        plt.subplot(len(sigmas), len(contractions), i)
        sm = results[(sigma, c)]
        plt.plot(traj2d[:,0], traj2d[:,1], alpha=0.3, label='raw')
        plt.plot(sm[:,0], sm[:,1], label='smoothed')
        plt.title(f"σ={sigma}, c={c}")
        plt.axis('equal')
        i += 1

plt.tight_layout()
plt.show()

## Lamphrodynamic Energy Heatmap

In [ ]:
class TinyRegressor(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(2, 16)
        self.fc2 = nn.Linear(16, 1)
    def forward(self, x):
        return self.fc2(torch.relu(self.fc1(x)))

model = TinyRegressor()
cfg = LamFlowConfig(lam_weight=1.0, gradient_weight=1.0, hessian_weight=0.0, use_hessian=False)
criterion = nn.MSELoss()
wrapped = LamFlowWrapper(model, criterion, cfg)

grid_x, grid_y = torch.meshgrid(torch.linspace(-10,10,200), torch.linspace(-10,10,200), indexing='ij')
points = torch.stack([grid_x, grid_y], dim=-1).reshape(-1,2)

with torch.no_grad():
    preds = wrapped.model(points).reshape(200,200)

plt.figure(figsize=(8,6))
plt.imshow(preds.detach().numpy(), extent=(-10,10,-10,10), origin='lower')
plt.colorbar(label='Model Output')
plt.title("Lamphrodynamic Model Output Heatmap")
plt.show()

## TARTAN vs Non‑TARTAN Trajectory Comparison

In [ ]:
cfg = TARTANConfig(aura_sigma=1.0, contraction_strength=0.1, buffer_size=128)
tartan = TARTANLayer(cfg)

smooth = tartan(traj2d.unsqueeze(1))[:,0,:].detach()

plt.figure(figsize=(10,4))

plt.subplot(1,2,1)
plt.plot(traj2d[:,0], traj2d[:,1])
plt.title("Raw Trajectory")
plt.axis('equal')

plt.subplot(1,2,2)
plt.plot(smooth[:,0], smooth[:,1])
plt.title("TARTAN Smoothed")
plt.axis('equal')

plt.tight_layout()
plt.show()